<a href="https://colab.research.google.com/github/Deepika2427/ITA0601-Machine-Learning/blob/main/Lab_3_and_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import math
from collections import Counter

# Step 1: Load the uploaded CSV file
df = pd.read_csv('/content/play_tennis.csv')

# Clean column names and text values
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print("Training Dataset:")
print(df.to_string(index=False))

# Step 2: Select the target column (last column)
target = df.columns[-1]
attributes = list(df.columns[:-1])

print("\nTarget column:", target)
print("Attributes:", attributes)

# Step 3: Calculate entropy
def entropy(values):
    counts = Counter(values)
    total = len(values)
    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result

# Step 4: Calculate information gain
def information_gain(data, attribute, target):
    total_entropy = entropy(data[target])
    weighted_entropy = 0

    for value in data[attribute].unique():
        subset = data[data[attribute] == value]
        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy

# Step 5: Build the ID3 decision tree
def id3(data, attributes, target):
    # All examples belong to the same class
    if data[target].nunique() == 1:
        return data[target].iloc[0]

    # No attributes left: return majority class
    if not attributes:
        return data[target].mode()[0]

    # Find attribute with maximum information gain
    gains = {
        attr: information_gain(data, attr, target)
        for attr in attributes
    }

    best_attribute = max(gains, key=gains.get)

    print("\nInformation Gain:", gains)
    print("Selected Attribute:", best_attribute)

    tree = {best_attribute: {}}
    remaining = [a for a in attributes if a != best_attribute]

    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]

        if subset.empty:
            tree[best_attribute][value] = data[target].mode()[0]
        else:
            tree[best_attribute][value] = id3(
                subset, remaining, target
            )

    return tree

# Step 6: Train the decision tree
tree = id3(df, attributes, target)

print("\nFinal Decision Tree:")
print(tree)

# Step 7: Predict a new sample
def predict(tree, sample, default):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample.get(attribute)

    if value not in tree[attribute]:
        return default

    return predict(tree[attribute][value], sample, default)

# Example new sample
new_sample = {
    attributes[0]: 'Sunny',
    attributes[1]: 'Cool',
    attributes[2]: 'High',
    attributes[3]: 'Strong'
}

prediction = predict(tree, new_sample, df[target].mode()[0])

print("\nNew Sample:")
print(new_sample)
print("Predicted Class:", prediction)


Training Dataset:
day  outlook temp humidity   wind play
 D1    Sunny  Hot     High   Weak   No
 D2    Sunny  Hot     High Strong   No
 D3 Overcast  Hot     High   Weak  Yes
 D4     Rain Mild     High   Weak  Yes
 D5     Rain Cool   Normal   Weak  Yes
 D6     Rain Cool   Normal Strong   No
 D7 Overcast Cool   Normal Strong  Yes
 D8    Sunny Mild     High   Weak   No
 D9    Sunny Cool   Normal   Weak  Yes
D10     Rain Mild   Normal   Weak  Yes
D11    Sunny Mild   Normal Strong  Yes
D12 Overcast Mild     High Strong  Yes
D13 Overcast  Hot   Normal   Weak  Yes
D14     Rain Mild     High Strong   No

Target column: play
Attributes: ['day', 'outlook', 'temp', 'humidity', 'wind']

Information Gain: {'day': 0.9402859586706311, 'outlook': 0.24674981977443933, 'temp': 0.02922256565895487, 'humidity': 0.15183550136234159, 'wind': 0.04812703040826949}
Selected Attribute: day

Final Decision Tree:
{'day': {'D1': 'No', 'D2': 'No', 'D3': 'Yes', 'D4': 'Yes', 'D5': 'Yes', 'D6': 'No', 'D7': 'Yes', 'D8'

Build an Artificial Neural Network by implementing the
Backpropagation algorithm and test the same using appropriate
data sets.# New Section

In [2]:

# Artificial Neural Network using Backpropagation

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print("Dataset loaded successfully")
print("Number of samples:", X.shape[0])
print("Number of input features:", X.shape[1])
print("Classes:", iris.target_names)

# Step 2: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 3: Normalize the input features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 4: Create the ANN model
# One hidden layer with 10 neurons
ann = MLPClassifier(
    hidden_layer_sizes=(10,),
    activation='relu',
    solver='sgd',
    learning_rate_init=0.01,
    max_iter=2000,
    random_state=42,
    tol=1e-5
)

# Step 5: Train the network using backpropagation
ann.fit(X_train, y_train)

print("\nANN training completed")

# Step 6: Test the network
y_pred = ann.predict(X_test)

print("\nActual Classes:")
print(y_test)

print("\nPredicted Classes:")
print(y_pred)

# Step 7: Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")

# Step 8: Display classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=iris.target_names
))

# Step 9: Test with a new sample
new_sample = np.array([[5.1, 3.5, 1.4, 0.2]])
new_sample_scaled = scaler.transform(new_sample)

prediction = ann.predict(new_sample_scaled)

print("\nNew Sample:", new_sample[0])
print("Predicted Flower:", iris.target_names[prediction[0]])


Dataset loaded successfully
Number of samples: 150
Number of input features: 4
Classes: ['setosa' 'versicolor' 'virginica']

ANN training completed

Actual Classes:
[0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]

Predicted Classes:
[0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 2 1 0 2 0]

Accuracy: 96.67 %

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


New Sample: [5.1 3.5 1.4 0.2]
Predicted Flower: setosa
